In [ ]:
import polyase as poly
import pandas as pd
import RNApysoforms as RNApy
import polars as pl

In [ ]:
poly.__version__

# Atlantic pairwise testing

## 1) Reading in the data 

In this section we will read in the data genespace output file, transcript quanitfication directory, tx2gene. 

In [ ]:
out_dir_syntelogfinder = "potato-allelic-orthogroups/output_atlantic_liftoff"
out_dir_long_read_quant = "nf-core-plantlongrnaseq/output_atlantic_liftoff_CDS2exon"
# File with syntelog categories from SyntelogFinder
var_obs_file = f"{out_dir_syntelogfinder}/genespace/unitato2Atl.no_scaffold.no_hap0_genespace_categories.tsv"
# Directory of gene counts from oarfish generated by long-read quantification pipeline
gene_counts_dir = f"{out_dir_long_read_quant}/oarfish/"
tx2gene = f"{out_dir_long_read_quant}/custom/combined.tx2gene.tsv"

### Dataset Information

**Source**: [Hoopes et al., 2022](https://doi.org/10.1016/j.molp.2022.01.003)

**Organism**: *Solanum tuberosum* cv. Atlantic (tetraploid)

**Experimental Design**:
- **10 samples** total from cultivar Atlantic
- **2 tissues**: Leaf and tuber
- **Biological replicates**: 5 samples per tissue type
- **Sequencing**: Long-read RNA-seq (ONT)
- **Quantification**: Oarfish transcript-level counts

**Sample Information**:

| Sample ID    | Tissue | Description |
|--------------|--------|-------------|
| SRR14993892  | Leaf   | Replicate 1 |
| SRR14993893  | Leaf   | Replicate 2 |
| SRR14993894  | Leaf   | Replicate 3 |
| SRR14993895  | Leaf   | Replicate 4 |
| SRR14996168  | Leaf   | Replicate 5 |
| SRR14995031  | Tuber  | Replicate 1 |
| SRR14995032  | Tuber  | Replicate 2 |
| SRR14995033  | Tuber  | Replicate 3 |
| SRR14995034  | Tuber  | Replicate 4 |
| SRR14995933  | Tuber  | Replicate 5 |

In [ ]:
# Define sample IDs and their conditions
sample_info = {
    # "SRR14993892": "leaf",
    # "SRR14993893": "leaf",
    "SRR14993894": "leaf",
    "SRR14993895": "leaf",
    "SRR14996168": "leaf",
    "SRR14995031": "tuber",
    "SRR14995032": "tuber",
    "SRR14995033": "tuber",
    "SRR14995034": "tuber",
    "SRR14995933": "tuber"
} 

# With isoforms
adata_isoform = poly.load_ase_data(
    var_obs_file=var_obs_file,
    sample_info=sample_info,
    tx_to_gene_file=tx2gene,
    isoform_counts_dir=gene_counts_dir,
    quant_dir=gene_counts_dir
)

### Add annotation

For interpretation of results we add functional gene annotations from an additional file to the adata.var

In [ ]:
# Load functional annotation data
#annotation_file = f"{input_dir}/Phureja_v4-v6.1_translations.xlsx"
annotation_file = "reference/ADAPT_liftoff/Phureja_v4-v6.1_translations.xlsx"
df_annotation = pd.read_excel(annotation_file)
df_annotation = df_annotation.iloc[:, 1:3]
df_annotation.columns = ['gene_id_unitato', 'functional_annotation']

# Remove version suffix from gene IDs (e.g., "GENE.1" -> "GENE")
adata_isoform.var['gene_id_unitato'] = adata_isoform.var['gene_id'].str.rsplit('.', n=1).str[0]
adata_isoform.var['original_index'] = adata_isoform.var.index

# Merge annotations
merged_data = pd.merge(
    adata_isoform.var,
    df_annotation,
    on='gene_id_unitato',
    how='left'
)

# Remove duplicates and restore index
merged_data = merged_data.drop_duplicates(subset=['original_index', 'gene_id'], keep='first')
merged_data.index = merged_data['original_index']
merged_data = merged_data.drop(columns=['original_index'])

# Update AnnData object
adata_isoform.var = merged_data

## 2) Gene-level analysis

In [ ]:
adata_gene = poly.aggregate_transcripts_to_genes(adata_isoform)

### A) Cis: Testing for differential allelic expression

We will test within one condition (here: 'leaf'), wether there are genes where alleles have unbalanced expression.

### Pairwiese is test


In [ ]:
#CPM threshold (automatically normalized)
# subset for conditions "leaf"

#CPM threshold (automatically normalized)

adata_gene_filtered = poly.filter_low_expressed_genes(
    adata_gene[(adata_gene.obs['condition'] == 'leaf'), :],
    min_expression=40.0,  # 20 counts
    mode= 'all',
    group_col ='Synt_id',  # group by gene_id to ensure filtering is done per gene
    #library_size_dependent=True, # this filters min_expression based on CPM of unique counts
    verbose=True)

adata_isoform_filtered = poly.calculate_allelic_ratios(adata_gene_filtered , 'unique_counts')
adata_isoform_filtered = poly.calculate_allelic_ratios(adata_gene_filtered , 'em_counts')
adata_isoform_filtered = poly.calculate_multi_ratios(adata_gene_filtered, 'unique_counts', 'ambiguous_counts')
# Test for differential allelic ratios withing conditions
# Filter for equal-length transcripts with equal transcript numbers
import numpy as np

var = adata_isoform_filtered.var

# restrict to synt_ids that already passed the tx-number check
sub = var

# genes whose mean_tx_length is constant across alleles
def const_length(x):
    return np.isclose(x.min(), x.max())

genes_same_length = sub.groupby('gene_id')['mean_tx_length'].apply(const_length)
valid_genes = genes_same_length[genes_same_length].index

mask = (var['gene_id'].isin(valid_genes) &
    (adata_gene_filtered.var["multimapping_ratio"] < 0.15) &
    adata_gene_filtered.var["synteny_category"].str.contains("[10]hap1_[10]hap2_[10]hap3_[10]hap4_s")
)

filtered = var[mask]
gene_counts = filtered.groupby('Synt_id')['gene_id'].nunique()
valid_synt_ids = gene_counts[gene_counts >= 2].index
gene_ids = filtered.loc[filtered['Synt_id'].isin(valid_synt_ids), 'gene_id'].unique()
       
adata_isoform_length_filtered = adata_isoform_filtered[:, mask].copy()



In [ ]:
cis_results_control = poly.test_allelic_ratios_pairwise(adata_isoform_length_filtered, 
                                                                 layer="unique_counts", 
                                                                 test_condition= "leaf", 
                                                                 inplace=True, ratio_diff_cutoff= 0.2)
# Get top differential syntelogs
cis_top_results = poly.get_top_differential_syntelogs(cis_results_control, 
                                                      n=45, sort_by='ratio_difference',
                                                      fdr_threshold=0.05, 
                                                      ratio_threshold=0.2)

In [ ]:
fig = poly.plot_top_differential_syntelogs_annotated(
    cis_results_control[cis_results_control['Synt_id'] == "Synt_id_10487"],
    n=12,
    plot_type='ratios',  # or 'ttest'
    figsize=(12,4), ratio_difference_threshold= 0.2
)    

In [ ]:
import numpy as np
cis_results_control["log2FC"] = cis_results_control.apply(
    lambda row: np.log2(row[f"cpm_allele_{row['allele_i']}_mean"] / row[f"cpm_allele_{row['allele_j']}_mean"])
    if row[f"cpm_allele_{row['allele_j']}_mean"] != 0 else np.nan,
    axis=1
)
# filter for genes with min log2FC 1 and FDR < 0.05
cis_results_control_filtered = cis_results_control[(cis_results_control['FDR'] < 0.05) & (cis_results_control['log2FC'].abs() > 1)] 
# save
cis_results_control_filtered.to_csv("LR_DESIREE_PAPER/ANALYSIS/ASE_results/comparison_DES_ATL/atlantic_pairwise_cis_results_significant.csv", index=False)
cis_results_control.to_csv("LR_DESIREE_PAPER/ANALYSIS/ASE_results/comparison_DES_ATL/atlantic_pairwise_cis_results_all.csv", index=False)
